<img src="images/logo.png" width="15%" height="15%">

# **LangChain Essentials**

### **7. Checkpointer**

Unless you use a **checkpointer**, your agent's state is reinitialized each time you invoke it.

When using a **checkpointer**, you identify each conversation with a **thread_id**, an arbitrary string used to group invocations under a common conversation.

It is part of **LangGraph**, the library upon which modern LangChain is built on.

We use the checkpointer for a simple memory system in this tutorial but this is much more powerful. You can use it to restate your agent at any execution step (time travel), useful in case of failure for example.

We'll build a web search tool using [Tavily](https://app.tavily.com/home). Grab your API key on their website, using their free tier.

In [ ]:
%pip install langchain langchain-mistralai python-dotenv langgraph tavily-python

In [1]:
from langchain.tools import tool
from dotenv import load_dotenv
from tavily import TavilyClient
from typing import Any
import os

load_dotenv()

# Research tool created by Gemini
@tool
def search_tavily(query: str, search_depth: str = "basic", max_results: int = 5) -> dict[str, Any]:
    """
    Searches the web using the Tavily API.
    
    Parameters:
    - query (str): The search query.
    - search_depth (str): 'basic' or 'advanced'. Advanced takes longer but is more thorough.
    - max_results (int): Number of results to return (default is 5).
    
    Returns:
    - dict: The raw JSON response from Tavily containing results.
    """
    # Retrieve the API key from environment variables
    api_key = os.getenv("TAVILY_API_KEY")
    
    if not api_key:
        raise ValueError("TAVILY_API_KEY environment variable not found. Please set it before running.")
    
    # Initialize the client
    tavily_client = TavilyClient(api_key=api_key)
    
    try:
        # Execute the search
        response = tavily_client.search(
            query=query, 
            search_depth=search_depth, 
            max_results=max_results
        )
        return response
    except Exception as e:
        print(f"An error occurred during the Tavily search: {e}")
        return {}

In [3]:
from langchain_mistralai import ChatMistralAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

llm = ChatMistralAI(model="mistral-small-latest")
llm_with_tools = llm.bind_tools([search_tavily])

agent = create_agent(
    model=llm_with_tools,
    tools=[search_tavily],
    system_prompt="You are an agent which can search for news on the Internet with its `search_tavily` tool",
    checkpointer=MemorySaver() # NEW
)

In [4]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "research1"}}

state = agent.invoke(
    {"messages": [HumanMessage(content="Summarize world news from June 22nd 2026")]},
    config=config
)

state

{'messages': [HumanMessage(content='Summarize world news from June 22nd 2026', additional_kwargs={}, response_metadata={}, id='dd03910f-d54f-4a20-8bab-9b5588074497'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'TnqpnqdZF', 'type': 'function', 'function': {'name': 'search_tavily', 'arguments': '{"query": "world news summary June 22nd 2026", "search_depth": "advanced", "max_results": 10}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 223, 'total_tokens': 263, 'completion_tokens': 40, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019efa0b-fd43-7c10-815a-f74292192de5-0', tool_calls=[{'name': 'search_tavily', 'args': {'query': 'world news summary June 22nd 2026', 'search_depth': 'advanced', 'max_results': 10}, 'id': 'TnqpnqdZF', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_toke

In [5]:
# Some models may include references in their response
for msg in state["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Summarize world news from June 22nd 2026
================================== Ai Message ==================================
Tool Calls:
  search_tavily (TnqpnqdZF)
 Call ID: TnqpnqdZF
  Args:
    query: world news summary June 22nd 2026
    search_depth: advanced
    max_results: 10
================================= Tool Message =================================
Name: search_tavily

{"query": "world news summary June 22nd 2026", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.democracynow.org/2026/6/22/headlines", "title": "Headlines for June 22, 2026 | Democracy Now!", "content": "## Under Intense U.S. Pressure, Cuban Lawmakers Approve Sweeping Economic Changes\n\nJun 22, 2026\n\nImage 46\n\n\n\nLink copied\n\nIn news from Cuba, lawmakers have passed sweeping economic changes that could lead to the privatization of much of Cuba’s socialist economy. The move comes

In [6]:
from IPython.display import Markdown, display

# Filter only for text (not references)
content = [chunk["text"] for chunk in state["messages"][-1].content if chunk["type"]=="text"]

# Display each chunk in markdown
for chunk in content:
    display(Markdown(chunk))

Here’s a summary of the major **world news events** from **June 22, 2026**:

---

### **International Relations & Diplomacy**
1. **U.S.-Iran Talks Show Progress**
   - Mediators from Pakistan and Qatar announced that the U.S. and Iran made **"encouraging progress"** during 18 hours of negotiations in Switzerland.
   - The two sides agreed to a **60-day roadmap** to finalize a deal aimed at ending the war in Iran, which began in late February.
   - The U.S. temporarily **lifted oil sanctions on Iran** to facilitate the talks.
   - Vice President **JD Vance** led the U.S. delegation, while Iran’s Parliament Speaker **Mohammad-Bagher Ghalibaf** headed the Iranian team.
   - Formal signing of a **memorandum of understanding (MOU)** was delayed due to "logistical issues," but optimism remains high

.

2. **Cuba’s Economic Reforms Under U.S. Pressure**
   - Cuba’s lawmakers approved **sweeping economic changes**, including the potential **privatization of much of its socialist economy**, amid intense U.S. pressure.
   - The U.S. has **blocked nearly all oil shipments** to Cuba and threatened to take over the country.
   - Prime Minister **Manuel Marrero Cruz** emphasized that the reforms are meant to **save socialism**, not abandon it.
   - Former Vice President **Ramiro Valdés Menéndez**, a key figure in the Cuban Revolution, died at age 94

.

3. **China-U.S. Trade War Escalates**
   - China imposed **export controls on 10 U.S. companies** and barred government procurement from 46 others in response to a U.S. blacklist targeting Chinese firms

.

4. **U.K. Political Crisis**
   - **Keir Starmer resigned as U.K. Prime Minister**, succumbing to political pressure. His departure has thrown the country’s political future into uncertainty

.

---

### **Conflict & Security**
5. **Mass Shootings in Montreal and the Philippines**
   - **Montreal, Canada**: A gunman killed a **police officer** at a hotel before being shot dead by police. A civilian was also killed

...

.

11. **Romania’s Political Crisis**
    - Romania’s parliament **rejected** the proposed government of Prime Minister-designate **Adrian Veștea**, leading to a new effort to form a government

.

---
### **Sports**
12. **World Cup Updates**
    - **Uzbekistan** made history by scoring its first-ever World Cup goal in a match against Colombia

.

---
### **Other Notable Events**
13. **Obama Presidential Center Opens**
    - Former President **Barack Obama** and Michelle Obama attended the opening of the **Obama Presidential Center in Chicago**, a massive and controversial monument

.

14. **U.S. Supreme Court Ruling**
    - The U.S. Supreme Court **reinstated a murder conviction** in the **Etan Patz case**, a decades-old high-profile case

.

---
### **Summary of Key Themes**
- **Diplomatic breakthroughs**: U.S.-Iran talks and easing of sanctions.
- **Political instability**: Resignations in the U.K. and Romania.
- **Violence and conflict**: Mass shootings, Ebola crisis, and escalation in Ukraine.
- **Economic and social changes**: Cuba’s reforms and global trade tensions.

In [7]:
state = agent.invoke(
    {"messages": [HumanMessage(content="Tell me more about this first topic")]},
    config=config
)

state

{'messages': [HumanMessage(content='Summarize world news from June 22nd 2026', additional_kwargs={}, response_metadata={}, id='dd03910f-d54f-4a20-8bab-9b5588074497'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'TnqpnqdZF', 'type': 'function', 'function': {'name': 'search_tavily', 'arguments': '{"query": "world news summary June 22nd 2026", "search_depth": "advanced", "max_results": 10}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 223, 'total_tokens': 263, 'completion_tokens': 40, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019efa0b-fd43-7c10-815a-f74292192de5-0', tool_calls=[{'name': 'search_tavily', 'args': {'query': 'world news summary June 22nd 2026', 'search_depth': 'advanced', 'max_results': 10}, 'id': 'TnqpnqdZF', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_toke